# Day 5 · Concept 2 of 5: Try Again, But Politely

**Big idea:** retry when a failure is temporary, but wait between tries and always stop after a few.
*About 15 minutes. No coding needed to follow along.*

---

### The idea in plain words
The card company line drops. You redial. If it drops again, you wait a bit longer before the next try, and you give up after three tries. You also do not redial at the exact second as everyone else.

That is **retry with backoff and jitter, with a limit**:
* **Backoff** = wait a little longer after each failure (0.2s, then 0.4s, then 0.8s...).
* **Jitter** = add a small random extra wait, so thousands of customers do not all redial at the same moment.
* **Limit** = a maximum number of tries. Never "retry forever".

### What went wrong in L2
In L2 we built a retry loop that stopped after a set number of tries, which was good. But it retried instantly, and it had no protection if the stop rule itself had a bug.

### What we do now (L3)
| Before (L2) | Now (L3) |
| --- | --- |
| Retry immediately | Wait longer after each failure, plus a little randomness |
| One stop rule | Two stop rules: our own limit **and** LangGraph's `recursion_limit` |

In [ ]:
# Step 0 - Connect to the AI model (Ollama). Run this cell first.
from langchain_ollama import ChatOllama

llm = ChatOllama(model="gpt-oss:120b-cloud", temperature=0)
try:
    print("Model says:", llm.invoke("Reply with just the word: ready").content)
except Exception as problem:
    print("The model is not reachable yet.")
    print("Fix: open a terminal, run   ollama signin   then restart this notebook.")
    print("Technical detail:", type(problem).__name__)

### The agent we will build
This time the graph has a **loop** (an arrow that goes back):

```
START -> [ call_tool ] --ok--------------> [ write_reply ] -> END
               ^   \--tries left--> [ wait ] --+
               |                              |
               +------------------------------+
               \--out of tries--> [ give_up ] -> END
```

The small function that chooses which arrow to follow is called the **router**. All of our retry policy lives in it.

In [ ]:
import time
from typing import TypedDict
from langgraph.graph import StateGraph, START, END

import random
import operator
from typing import Annotated

MAX_TRIES = 3        # the limit: never more than 3 phone calls
BASE_WAIT = 0.2      # first wait; it doubles each time (0.2, 0.4, ...)

class State(TypedDict):
    script: list                          # pretend outcomes for each call, e.g. ["error", "ok"]
    attempt: int
    error: str
    balance: str
    reply: str
    log: Annotated[list, operator.add]    # a diary that every box can add lines to

def call_tool(state):
    attempt = state["attempt"] + 1
    outcome = state["script"][attempt - 1]         # we decide in advance what happens: "ok" or "error"
    if outcome == "error":
        return {"attempt": attempt, "error": "ConnectionError", "log": [f"try {attempt}: the line dropped"]}
    return {"attempt": attempt, "error": "", "balance": "1,250 INR", "log": [f"try {attempt}: success"]}

def wait(state):
    pause = BASE_WAIT * 2 ** (state["attempt"] - 1)      # backoff: 0.2, 0.4, 0.8 ...
    pause += random.uniform(0, 0.1)                      # jitter: a little randomness
    time.sleep(pause)
    return {"log": [f"waited {pause:.2f}s before trying again"]}

def give_up(state):
    return {"reply": "Sorry, we could not reach the bank. Please try again later.",
            "log": [f"gave up after {state['attempt']} tries"]}

def write_reply(state):
    return {"reply": llm.invoke("You are a polite bank assistant. Say this in one short sentence: the balance is " + state["balance"]).content}

def router(state):       # <- the whole retry policy is these four lines
    if state["error"] == "":
        return "done"
    if state["attempt"] < MAX_TRIES:
        return "retry"
    return "give_up"

builder = StateGraph(State)
builder.add_node("call_tool", call_tool)
builder.add_node("wait", wait)
builder.add_node("give_up", give_up)
builder.add_node("write_reply", write_reply)
builder.add_edge(START, "call_tool")
builder.add_conditional_edges("call_tool", router, {"done": "write_reply", "retry": "wait", "give_up": "give_up"})
builder.add_edge("wait", "call_tool")           # the arrow that goes back = the loop
builder.add_edge("write_reply", END)
builder.add_edge("give_up", END)
agent = builder.compile()

def run(script):
    result = agent.invoke({"script": script, "attempt": 0, "error": "", "balance": "", "reply": "", "log": []})
    for line in result["log"]:
        print("  ", line)
    print("   Customer sees:", result["reply"])

### Run it: a flaky line, then a dead line
`script` is our pretend list of what happens on each call.

In [ ]:
print("Flaky line: one failure, then success")
run(["error", "ok"])
print()
print("Dead line: every call fails")
run(["error", "error", "error"])

**What you should see:** the flaky line recovers on try 2 after a short wait. The dead line stops after exactly **3 tries**, and the customer gets a clear message. It never loops forever.

---
### Why jitter? A picture in numbers
Five customers hit the same outage at the same moment. Look at *when* each one redials.

In [ ]:
random.seed(7)
print("Without jitter, 5 customers redial at:", [0.2, 0.2, 0.2, 0.2, 0.2], "  <- all at once, outage gets worse")
print("With jitter,    5 customers redial at:", sorted(round(0.2 + random.uniform(0, 0.2), 2) for _ in range(5)), "  <- spread out")

---
### The second stop rule: the seat belt
What if someone writes a **buggy router** that always says "retry"? Our own limit would never trigger. LangGraph has a built-in emergency brake called `recursion_limit`: a hard cap on how many steps one run may take.

We build a deliberately broken agent to see the brake work.

In [ ]:
from langgraph.errors import GraphRecursionError

def buggy_router(state):
    return "retry"            # BUG: never says done, never gives up

broken = StateGraph(State)
broken.add_node("call_tool", call_tool)
broken.add_node("wait", wait)
broken.add_edge(START, "call_tool")
broken.add_conditional_edges("call_tool", buggy_router, {"retry": "wait"})
broken.add_edge("wait", "call_tool")
broken_agent = broken.compile()

try:
    broken_agent.invoke({"script": ["error"] * 50, "attempt": 0, "error": "", "balance": "", "reply": "", "log": []},
                        config={"recursion_limit": 6})
except GraphRecursionError:
    print("Stopped by LangGraph's emergency brake after 6 steps. The bug did not loop forever.")

### Your turn: predict first, then run
Set `MAX_TRIES = 2` and run the script `["error", "error", "ok"]` (two failures, then success).

**Predict:** does the customer get a balance?

In [ ]:
MAX_TRIES = 2
run(["error", "error", "ok"])
MAX_TRIES = 3         # put the rule back

**Answer:** no. Two tries are used up before the success would have happened, so the agent gives up. Retries are a *budget*, not a promise.

---

### Remember
* Retry only **temporary** problems, wait longer each time, add a little randomness, and stop after a few tries.
* Keep **two** stop rules: your own limit, and the framework's `recursion_limit` as a seat belt.

> **Technical note (optional):** in Day 19 `agent.py::run_tool` folds this cycle into one loop with `MCP_ATTEMPTS = 2`. Here we draw it as graph boxes so you can see it.

**Next notebook:** what should the agent say when retries run out, and what if retrying can *never* help?